
 04 - IMDb - Camada Prata

MVP-3 — Engenharia de Dados

 Este notebook transforma os dados IMDb da camada Bronze
 em uma representação limpa e validada na camada Prata.

Achados do profiling
 - 50.000 registros na Bronze;
 - 418 registros duplicados;
 - nenhum valor nulo em review;
 - nenhum valor nulo em sentiment;
 - nenhuma review vazia;
 - 25.000 avaliações positive;
 - 25.000 avaliações negative.

ransformações
 - remoção de duplicidades;
 - padronização do sentimento;
 - criação de identificador técnico;
 - preservação da rastreabilidade da origem;
 - persistência em formato Delta.

In [0]:
from pyspark.sql import functions as F


CATALOGO = "mvp3_eng_dados"

SCHEMA_BRONZE = "bronze"
SCHEMA_PRATA = "prata"


TABELA_BRONZE = (
    f"{CATALOGO}.{SCHEMA_BRONZE}.imdb_avaliacoes"
)

TABELA_PRATA = (
    f"{CATALOGO}.{SCHEMA_PRATA}.imdb_avaliacoes"
)


print("✓ Configuração carregada")
print(f"Origem:  {TABELA_BRONZE}")
print(f"Destino: {TABELA_PRATA}")

✓ Configuração carregada
Origem:  mvp3_eng_dados.bronze.imdb_avaliacoes
Destino: mvp3_eng_dados.prata.imdb_avaliacoes


In [0]:
df_imdb_bronze = spark.table(
    TABELA_BRONZE
)


qtd_bronze = df_imdb_bronze.count()


print(
    f"✓ IMDb Bronze carregado: "
    f"{qtd_bronze:,} registros"
)

✓ IMDb Bronze carregado: 50,000 registros


In [0]:
df_imdb_negocio = (
    df_imdb_bronze
    .select(
        "review",
        "sentiment"
    )
)


print("✓ Campos de negócio selecionados")

✓ Campos de negócio selecionados


In [0]:
df_imdb_sem_duplicados = (
    df_imdb_negocio
    .dropDuplicates(
        [
            "review",
            "sentiment"
        ]
    )
)


qtd_sem_duplicados = (
    df_imdb_sem_duplicados.count()
)


duplicados_removidos = (
    qtd_bronze
    - qtd_sem_duplicados
)


print(
    f"Registros Bronze: "
    f"{qtd_bronze:,}"
)

print(
    f"Registros após deduplicação: "
    f"{qtd_sem_duplicados:,}"
)

print(
    f"Duplicados removidos: "
    f"{duplicados_removidos:,}"
)

Registros Bronze: 50,000
Registros após deduplicação: 49,582
Duplicados removidos: 418


In [0]:
df_imdb_padronizado = (
    df_imdb_sem_duplicados
    .withColumn(
        "sentimento",
        F.when(
            F.lower(
                F.trim(
                    F.col("sentiment")
                )
            ) == "positive",
            F.lit("POSITIVO")
        )
        .when(
            F.lower(
                F.trim(
                    F.col("sentiment")
                )
            ) == "negative",
            F.lit("NEGATIVO")
        )
        .otherwise(
            F.lit("DESCONHECIDO")
        )
    )
)


print("✓ Sentimento padronizado")

✓ Sentimento padronizado


In [0]:
df_imdb_prata = (
    df_imdb_padronizado
    .withColumn(
        "id_avaliacao",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("review"),
                F.col("sentiment")
            ),
            256
        )
    )
    .withColumn(
        "fonte_dados",
        F.lit("IMDb")
    )
    .withColumn(
        "data_hora_processamento",
        F.current_timestamp()
    )
)

In [0]:
df_imdb_prata = (
    df_imdb_prata
    .select(
        "id_avaliacao",
        "review",
        "sentiment",
        "sentimento",
        "fonte_dados",
        "data_hora_processamento"
    )
)


df_imdb_prata.printSchema()

root
 |-- id_avaliacao: string (nullable = true)
 |-- review: string (nullable = true)
 |-- sentiment: string (nullable = true)
 |-- sentimento: string (nullable = false)
 |-- fonte_dados: string (nullable = false)
 |-- data_hora_processamento: timestamp (nullable = false)



In [0]:
qtd_nulos_review = (
    df_imdb_prata
    .filter(
        F.col("review").isNull()
    )
    .count()
)


qtd_nulos_sentimento = (
    df_imdb_prata
    .filter(
        F.col("sentimento").isNull()
    )
    .count()
)


qtd_desconhecidos = (
    df_imdb_prata
    .filter(
        F.col("sentimento")
        == "DESCONHECIDO"
    )
    .count()
)


qtd_ids_distintos = (
    df_imdb_prata
    .select("id_avaliacao")
    .distinct()
    .count()
)


print(
    f"Reviews nulas: "
    f"{qtd_nulos_review:,}"
)

print(
    f"Sentimentos nulos: "
    f"{qtd_nulos_sentimento:,}"
)

print(
    f"Sentimentos desconhecidos: "
    f"{qtd_desconhecidos:,}"
)

print(
    f"IDs distintos: "
    f"{qtd_ids_distintos:,}"
)

Reviews nulas: 0
Sentimentos nulos: 0
Sentimentos desconhecidos: 0
IDs distintos: 49,582


In [0]:
assert qtd_nulos_review == 0, (
    "Existem reviews nulas."
)

assert qtd_nulos_sentimento == 0, (
    "Existem sentimentos nulos."
)

assert qtd_desconhecidos == 0, (
    "Existem valores de sentimento não mapeados."
)

assert qtd_ids_distintos == qtd_sem_duplicados, (
    "Existem IDs de avaliação duplicados."
)


print("✓ Validações de qualidade aprovadas")

✓ Validações de qualidade aprovadas


In [0]:
display(
    df_imdb_prata
    .groupBy(
        "sentiment",
        "sentimento"
    )
    .count()
    .orderBy(
        "sentimento"
    )
)

sentiment,sentimento,count
negative,NEGATIVO,24698
positive,POSITIVO,24884


In [0]:
(
    df_imdb_prata.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        TABELA_PRATA
    )
)


print(
    f"✓ Tabela criada: "
    f"{TABELA_PRATA}"
)

✓ Tabela criada: mvp3_eng_dados.prata.imdb_avaliacoes


In [0]:
assert spark.catalog.tableExists(
    TABELA_PRATA
), "Tabela Prata não encontrada."


df_imdb_prata_validacao = (
    spark.table(
        TABELA_PRATA
    )
)


qtd_prata = (
    df_imdb_prata_validacao.count()
)


print(
    f"Bronze: "
    f"{qtd_bronze:,}"
)

print(
    f"Prata:  "
    f"{qtd_prata:,}"
)

print(
    f"Removidos por duplicidade: "
    f"{qtd_bronze - qtd_prata:,}"
)

Bronze: 50,000
Prata:  49,582
Removidos por duplicidade: 418


In [0]:
display(
    df_imdb_prata_validacao
    .select(
        "id_avaliacao",
        "review",
        "sentiment",
        "sentimento",
        "fonte_dados"
    )
    .limit(10)
)

id_avaliacao,review,sentiment,sentimento,fonte_dados
2230317909750eaa1c15cd45ecea8e78c8055b0624a90e6ccb63b61a6a92ff3b,"I sure would like to see a resurrection of a up dated Seahunt series with the tech they have today it would bring back the kid excitement in me.I grew up on black and white TV and Seahunt with Gunsmoke were my hero's every week.You have my vote for a comeback of a new sea hunt.We need a change of pace in TV and this would work for a world of under water adventure.Oh by the way thank you for an outlet like this to view many viewpoints about TV and the many movies.So any ole way I believe I've got what I wanna say.Would be nice to read some more plus points about sea hunt.If my rhymes would be 10 lines would you let me submit,or leave me out to be in doubt and have me to quit,If this is so then I must go so lets do it.",positive,POSITIVO,IMDb
3848c3c3a470906ed5679e658a6ab1b828eba54f14c7b8d023d9a0bb1ffa4a9f,"Taut and organically gripping, Edward Dmytryk's Crossfire is a distinctive suspense thriller, an unlikely ""message"" movie using the look and devices of the noir cycle.Bivouacked in Washington, DC, a company of soldiers cope with their restlessness by hanging out in bars. Three of them end up at a stranger's apartment where Robert Ryan, drunk and belligerent, beats their host (Sam Levene) to death because he happens to be Jewish. Police detective Robert Young investigates with the help of Robert Mitchum, who's assigned to Ryan's outfit. Suspicion falls on the second of the three (George Cooper), who has vanished. Ryan slays the third buddy (Steve Brodie) to insure his silence before Young closes in.Abetted by a superior script by John Paxton, Dmytryk draws precise performances from his three starring Bobs. Ryan, naturally, does his prototypical Angry White Male (and to the hilt), while Mitchum underplays with his characteristic alert nonchalance (his role, however, is not central); Young may never have been better. Gloria Grahame gives her first fully-fledged rendition of the smart-mouthed, vulnerable tramp, and, as a sad sack who's leeched into her life, Paul Kelly haunts us in a small, peripheral role that he makes memorable.The politically engaged Dmytryk perhaps inevitably succumbs to sermonizing, but it's pretty much confined to Young's reminiscence of how his Irish grandfather died at the hands of bigots a century earlier (thus, incidentally, stretching chronology to the limit). At least there's no attempt to render an explanation, however glib, of why Ryan hates Jews (and hillbillies and...).Curiously, Crossfire survives even the major change wrought upon it -- the novel it's based on (Richard Brooks' The Brick Foxhole) dealt with a gay-bashing murder. But homosexuality in 1947 was still Beyond The Pale. News of the Holocaust had, however, begun to emerge from the ashes of Europe, so Hollywood felt emboldened to register its protest against anti-Semitism (the studios always quaked at the prospect of offending any potential ticket buyer).But while the change from homophobia to anti-Semitism works in general, the specifics don't fit so smoothly. The victim's chatting up a lonesome, drunk young soldier then inviting him back home looks odd, even though (or especially since) there's a girlfriend in tow. It raises the question whether this scenario was retained inadvertently or left in as a discreet tip-off to the original engine generating Ryan's murderous rage.",positive,POSITIVO,IMDb
668b0fdb833a6b971796ef5e0fd0d12f4d1a20050b144b0a7add1c756af8412e,"Nicholas Walker is Paul, the local town Reverand who's married to Martha (Ally Sheedy), but also is a habitual womanizer and decides to fake his own death to run away with his current affair, Veronica (Dara Tomanovich). However in so doing, he gets a bout of amnesia (hence the name of the film). Sally Kirkland is also on hand as a crazy old coot who pines for the good Reverand in a shades of ""Misery"" type of way. It's sad to see a pretty good cast wasted li

In [0]:
print("=" * 60)
print("IMDb - CAMADA PRATA CONCLUÍDA")
print("=" * 60)


print("\nOrigem:")
print(TABELA_BRONZE)


print("\nDestino:")
print(TABELA_PRATA)


print("\nResultado:")

print(
    f"✓ Registros Bronze: "
    f"{qtd_bronze:,}"
)

print(
    f"✓ Duplicados removidos: "
    f"{duplicados_removidos:,}"
)

print(
    f"✓ Registros Prata: "
    f"{qtd_prata:,}"
)


print("\nTransformações:")

print("✓ Duplicidades removidas")
print("✓ Sentimento padronizado")
print("✓ Valor original de sentiment preservado")
print("✓ Identificador técnico criado")
print("✓ Fonte identificada")
print("✓ Validações de qualidade executadas")


print("\nStatus:")
print("✓ IMDb PRATA VALIDADA")


print("\nPróxima etapa:")
print("05_rt_filmes_prata")

IMDb - CAMADA PRATA CONCLUÍDA

Origem:
mvp3_eng_dados.bronze.imdb_avaliacoes

Destino:
mvp3_eng_dados.prata.imdb_avaliacoes

Resultado:
✓ Registros Bronze: 50,000
✓ Duplicados removidos: 418
✓ Registros Prata: 49,582

Transformações:
✓ Duplicidades removidas
✓ Sentimento padronizado
✓ Valor original de sentiment preservado
✓ Identificador técnico criado
✓ Fonte identificada
✓ Validações de qualidade executadas

Status:
✓ IMDb PRATA VALIDADA

Próxima etapa:
05_rt_filmes_prata


####Observações:
- Um aspecto importante deste desenho é que não estamos alterando review — não faremos tratamento de maiúsculas/minúsculas, remoção de pontuação, artigos, preposições, conjunções, stemming etc. Isso seria preparação específica para NLP/modelagem e poderia destruir informação textual útil. 
- Neste pipeline de Engenharia de Dados, a Prata preservará o texto original limpo estruturalmente.
- Também não criaremos agregações. Elas pertencem à etapa Ouro. A documentação do Databricks (https://docs.databricks.com/aws/pt/lakehouse/medallion) caracteriza justamente a Prata como camada de dados validados/deduplicados e a Ouro como camada voltada à modelagem e agregação para consumo